In [2]:
!pip install datasets -q

import sys
sys.path.append('/kaggle/working')

In [3]:
# invariant-core/data/loaders/xnli_loader.py

from datasets import load_dataset
import pandas as pd
from collections import defaultdict

# Наши 8 сред: регион → языки XNLI
REGION_LANGUAGES = {
    "eastern_europe": ["ru", "bg"],        # болгарский есть в XNLI
    "western_europe": ["de", "fr", "es"],
    "anglosphere":    ["en"],
    "arab_world":     ["ar"],
    "south_asia":     ["hi", "ur"],
    "east_asia":      ["zh"],
    "southeast_asia": ["th", "sw"],        # суахили как прокси Африки
    "latin_america":  ["es"],              # дублирует, но с другим split
}

# XNLI содержит 15 языков: ar, bg, de, el, en, es, fr, hi,
# ru, sw, th, tr, ur, vi, zh

def load_xnli_by_region(split="validation", samples_per_lang=5000):
    """
    Загружает XNLI, группирует по регионам-средам.
    split: "train" | "validation" | "test"
    samples_per_lang: сколько примеров брать на язык
    """

    # XNLI загружается отдельно на каждый язык
    region_data = defaultdict(list)
    lang_stats = {}

    for region, languages in REGION_LANGUAGES.items():
        print(f"\n→ Загрузка региона: {region}")

        for lang in languages:
            try:
                # Загружаем датасет для языка
                dataset = load_dataset(
                    "xnli",
                    lang,
                    split=split,
                )

                # Берём нужное количество
                n = min(samples_per_lang, len(dataset))
                subset = dataset.select(range(n))

                # Добавляем метаданные
                for item in subset:
                    region_data[region].append({
                        "premise":    item["premise"],
                        "hypothesis": item["hypothesis"],
                        "label":      item["label"],  # 0=entailment, 1=neutral, 2=contradiction
                        "language":   lang,
                        "region":     region,
                        # Позже добавим: source_type, time_period
                    })

                lang_stats[lang] = n
                print(f"  ✓ {lang}: {n} примеров")

            except Exception as e:
                print(f"  ✗ {lang}: ошибка — {e}")

    return region_data, lang_stats


def region_data_to_df(region_data):
    """Конвертирует в DataFrame для удобного анализа."""
    all_rows = []
    for region, items in region_data.items():
        all_rows.extend(items)
    return pd.DataFrame(all_rows)


def print_stats(df):
    """Базовая статистика по средам."""
    print("\n=== СТАТИСТИКА ПО СРЕДАМ ===")
    print(df.groupby("region")["label"].value_counts().unstack(fill_value=0))
    print(f"\nВсего примеров: {len(df)}")
    print(f"Языков: {df['language'].nunique()}")
    print(f"Регионов: {df['region'].nunique()}")

    # Баланс — важно для IRM
    sizes = df.groupby("region").size()
    print(f"\nРазмеры сред (мин/макс): {sizes.min()} / {sizes.max()}")
    ratio = sizes.max() / sizes.min()
    if ratio > 3:
        print(f"⚠ Дисбаланс {ratio:.1f}x — нужна стратификация")
    else:
        print(f"✓ Баланс в норме ({ratio:.1f}x)")


if __name__ == "__main__":
    print("Загружаем XNLI validation split...")
    region_data, lang_stats = load_xnli_by_region(
        split="validation",
        samples_per_lang=5000
    )

    df = region_data_to_df(region_data)
    print_stats(df)

    # Сохраняем
    df.to_parquet("xnli_by_region.parquet", index=False)
    print("\n✓ Сохранено: xnli_by_region.parquet")

Загружаем XNLI validation split...

→ Загрузка региона: eastern_europe


README.md: 0.00B [00:00, ?B/s]

ru/train-00000-of-00001.parquet:   0%|          | 0.00/70.0M [00:00<?, ?B/s]

ru/test-00000-of-00001.parquet:   0%|          | 0.00/477k [00:00<?, ?B/s]

ru/validation-00000-of-00001.parquet:   0%|          | 0.00/239k [00:00<?, ?B/s]

...

sw/validation-00000-of-00001.parquet:   0%|          | 0.00/158k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

  ✓ sw: 2490 примеров

→ Загрузка региона: latin_america
  ✓ es: 2490 примеров

=== СТАТИСТИКА ПО СРЕДАМ ===
label              0     1     2
region                          
anglosphere      830   830   830
arab_world       830   830   830
east_asia        830   830   830
eastern_europe  1660  1660  1660
latin_america    830   830   830
south_asia      1660  1660  1660
southeast_asia  1660  1660  1660
western_europe  2490  2490  2490

Всего примеров: 32370
Языков: 12
Регионов: 8

Размеры сред (мин/макс): 2490 / 7470
✓ Баланс в норме (3.0x)

✓ Сохранено: xnli_by_region.parquet


In [4]:
region_data, stats = load_xnli_by_region(split="validation", samples_per_lang=2000)
df = region_data_to_df(region_data)
print_stats(df)

# Посмотреть несколько примеров
df.sample(5)


→ Загрузка региона: eastern_europe
  ✓ ru: 2000 примеров
  ✓ bg: 2000 примеров

→ Загрузка региона: western_europe
  ✓ de: 2000 примеров
  ✓ fr: 2000 примеров
  ✓ es: 2000 примеров

→ Загрузка региона: anglosphere
  ✓ en: 2000 примеров

→ Загрузка региона: arab_world
  ✓ ar: 2000 примеров

→ Загрузка региона: south_asia
  ✓ hi: 2000 примеров
  ✓ ur: 2000 примеров

→ Загрузка региона: east_asia
  ✓ zh: 2000 примеров

→ Загрузка региона: southeast_asia
  ✓ th: 2000 примеров
  ✓ sw: 2000 примеров

→ Загрузка региона: latin_america
  ✓ es: 2000 примеров

=== СТАТИСТИКА ПО СРЕДАМ ===
label              0     1     2
region                          
anglosphere      666   667   667
arab_world       666   667   667
east_asia        666   667   667
eastern_europe  1332  1334  1334
latin_america    666   667   667
south_asia      1332  1334  1334
southeast_asia  1332  1334  1334
western_europe  1998  2001  2001

Всего примеров: 26000
Языков: 12
Регионов: 8

Размеры сред (мин/макс): 2000 / 6000

,premise,hypothesis,label,language,region
6004,"Je ne savais pas dans quoi je me lançais, donc...",Je savais exactement ce que j'avais à faire qu...,2,fr,western_europe
12436,اوه- حسنا وداعا الآن,كان من الرائع التحدث إليك وسأتحدث معك غداً.,1,ar,arab_world
14370,"mujhe nahi pata, thik hai, aapse baat karke ac...",मुझे लगता है कि मुझे पता है कि मैं ठीक क्यों न...,1,hi,south_asia
21579,ระบบคัดกรองที่พรมแดนควรจะคัดกรองบุคคลอย่างมีปร...,เราไม่มีเวลาสำหรับความสะดวกสบายเมื่อจัดการเรื่...,2,th,southeast_asia
19038,"崎岖的Serra de Tramuntana山脉被扫到了海面上, 这里特别陡峭, 有很少的出...",山上时常有滑坡现象导致建立海港比较困难。,1,zh,east_asia


In [5]:
# Ячейка 4 — детектор противоречий

def extract_contradictions(df):
    """
    Выделяем contradiction-пары (label=2) по каждому региону.
    Это основа для оси декларация vs реальность.
    """
    contradictions = df[df["label"] == 2].copy()
    
    print("=== ПРОТИВОРЕЧИЯ ПО РЕГИОНАМ ===")
    stats = contradictions.groupby("region").size()
    print(stats)
    print(f"\nВсего contradiction-пар: {len(contradictions)}")
    print(f"Это {len(contradictions)/len(df)*100:.1f}% от всех данных")
    
    # Смотрим примеры из каждого региона
    print("\n=== ПРИМЕРЫ ПРОТИВОРЕЧИЙ ===")
    for region in df["region"].unique():
        sample = contradictions[contradictions["region"] == region].sample(1).iloc[0]
        print(f"\n[{region} / {sample['language']}]")
        print(f"  Утверждение: {sample['premise'][:100]}...")
        print(f"  Противоречие: {sample['hypothesis'][:100]}...")
    
    return contradictions

contradictions = extract_contradictions(df)

# Сохраняем отдельно
contradictions.to_parquet("contradictions_by_region.parquet", index=False)
print("\n✓ Сохранено: contradictions_by_region.parquet")

=== ПРОТИВОРЕЧИЯ ПО РЕГИОНАМ ===
region
anglosphere        667
arab_world         667
east_asia          667
eastern_europe    1334
latin_america      667
south_asia        1334
southeast_asia    1334
western_europe    2001
dtype: int64

Всего contradiction-пар: 8671
Это 33.4% от всех данных

=== ПРИМЕРЫ ПРОТИВОРЕЧИЙ ===

[eastern_europe / bg]
  Утверждение: Да, винаги съм казвал, че ако умра ще се върна като куче, това ще е най-хубавото, което може да се с...
  Противоречие: Не вярвам в никакъв вид задгробен живот....

[western_europe / fr]
  Утверждение: Plus de 500 millions de personnes traversent annuellement les frontières américaines aux points d'en...
  Противоречие: Seulement 20,000 personnes traversent la frontière chaque année....

[anglosphere / en]
  Утверждение: Henceforth, national unity always played second fiddle to ethnic, religious, and above all economic ...
  Противоречие: National unity doesn't matter at all....

[arab_world / ar]
  Утверждение: خلف منطقة أمريكا ال

In [6]:
# Ячейка 5 — расширенные маркеры

import re

DECLARATION_MARKERS = {
    "ru": ["должен", "нужно", "следует", "считаю", "полагаю", "верю",
           "мы всегда", "принято", "нельзя", "обязан", "важно", "необходимо",
           "правильно", "неправильно", "хорошо", "плохо", "лучше", "хуже",
           "надо", "стоит", "желательно", "запрещено", "разрешено"],
    "bg": ["трябва", "вярвам", "винаги", "никога", "трябва да", "важно",
           "необходимо", "правилно", "погрешно"],
    "en": ["should", "must", "believe", "think", "always", "never",
           "supposed to", "ought", "consider", "feel that", "important",
           "necessary", "right", "wrong", "better", "worse", "need to",
           "have to", "it is good", "it is bad", "we should", "one must"],
    "de": ["sollte", "muss", "glaube", "denke", "immer", "nie", "wichtig",
           "notwendig", "richtig", "falsch", "besser", "schlechter", "müssen",
           "dürfen", "sollen"],
    "fr": ["devrait", "doit", "crois", "pense", "toujours", "jamais",
           "faut", "important", "nécessaire", "correct", "incorrect",
           "mieux", "pire", "on doit", "il faut"],
    "es": ["debe", "hay que", "creo", "pienso", "siempre", "nunca",
           "debería", "importante", "necesario", "correcto", "incorrecto",
           "mejor", "peor", "se debe", "es necesario", "es importante"],
    "ar": ["يجب", "ينبغي", "أعتقد", "دائما", "أبدا", "مهم", "ضروري",
           "صحيح", "خاطئ", "أفضل", "أسوأ", "لا بد", "من المهم"],
    "hi": ["चाहिए", "मानता", "हमेशा", "कभी नहीं", "जरूरी", "महत्वपूर्ण",
           "सही", "गलत", "बेहतर", "बुरा", "होना चाहिए"],
    "ur": ["چاہیے", "مانتا", "ہمیشہ", "کبھی نہیں", "ضروری", "اہم",
           "صحیح", "غلط", "بہتر", "برا"],
    "zh": ["应该", "必须", "认为", "总是", "从不", "重要", "必要",
           "正确", "错误", "更好", "更坏", "需要", "应当"],
    "th": ["ควร", "ต้อง", "เชื่อ", "คิด", "เสมอ", "ไม่เคย", "สำคัญ",
           "จำเป็น", "ถูก", "ผิด", "ดีกว่า", "แย่กว่า"],
    "sw": ["lazima", "inabidi", "nadhani", "daima", "kamwe", "muhimu",
           "sahihi", "vibaya", "bora", "mbaya zaidi", "inapaswa"],
}

BEHAVIOR_MARKERS = {
    "ru": ["сделал", "произошло", "составило", "было", "стало", "в результате",
           "по данным", "согласно", "статистика", "исследование показало",
           "выяснилось", "обнаружено", "зафиксировано", "процент", "число",
           "количество", "увеличилось", "уменьшилось", "погиб", "арестован"],
    "bg": ["се случи", "стана", "беше", "според", "статистика", "процент",
           "брой", "увеличи", "намали"],
    "en": ["happened", "occurred", "resulted", "according to", "statistics",
           "data shows", "was found", "percent", "number of", "increased",
           "decreased", "was arrested", "was killed", "study found",
           "research shows", "survey", "poll", "reported", "confirmed"],
    "de": ["geschah", "ergab", "laut", "Statistik", "Prozent", "Anzahl",
           "stieg", "sank", "wurde verhaftet", "Studie", "Forschung"],
    "fr": ["s'est passé", "selon", "statistiques", "résultat", "données",
           "pourcentage", "nombre", "augmenté", "diminué", "étude", "rapport"],
    "es": ["ocurrió", "según", "estadísticas", "resultado", "datos",
           "porcentaje", "número", "aumentó", "disminuyó", "estudio",
           "informe", "fue arrestado", "murió"],
    "ar": ["حدث", "وفقا", "إحصائيات", "نتيجة", "بيانات", "نسبة",
           "عدد", "ازداد", "انخفض", "دراسة", "تقرير"],
    "hi": ["हुआ", "के अनुसार", "आंकड़े", "परिणाम", "प्रतिशत", "संख्या",
           "बढ़ा", "घटा", "अध्ययन", "रिपोर्ट"],
    "ur": ["ہوا", "کے مطابق", "اعداد", "نتیجہ", "فیصد", "تعداد",
           "بڑھا", "کم ہوا", "تحقیق", "رپورٹ"],
    "zh": ["发生", "根据", "统计", "结果", "数据", "百分", "数量",
           "增加", "减少", "研究", "报告", "被捕", "死亡"],
    "th": ["เกิดขึ้น", "ตาม", "สถิติ", "ผล", "ข้อมูล", "ร้อยละ",
           "จำนวน", "เพิ่มขึ้น", "ลดลง", "การศึกษา", "รายงาน"],
    "sw": ["ilitokea", "kulingana", "takwimu", "matokeo", "data",
           "asilimia", "idadi", "iliongezeka", "ilipungua", "utafiti",
           "ripoti", "alikamatwa"],
}

def detect_source_type(text, language):
    text_lower = text.lower()
    decl_markers  = DECLARATION_MARKERS.get(language, DECLARATION_MARKERS["en"])
    behav_markers = BEHAVIOR_MARKERS.get(language, BEHAVIOR_MARKERS["en"])
    decl_score  = sum(1 for m in decl_markers  if m in text_lower)
    behav_score = sum(1 for m in behav_markers if m in text_lower)
    if decl_score > behav_score:
        return "declaration"
    elif behav_score > decl_score:
        return "behavior"
    else:
        return "neutral"

def add_source_type(df):
    df = df.copy()
    df["source_type"] = df.apply(
        lambda row: detect_source_type(row["premise"], row["language"]),
        axis=1
    )
    return df

# Загружаем сохранённый файл (чтобы не перекачивать XNLI)
df = pd.read_parquet("xnli_by_region.parquet")
df = add_source_type(df)

print("=== ТИПЫ ИСТОЧНИКОВ ПО РЕГИОНАМ ===")
print(df.groupby(["region", "source_type"]).size().unstack(fill_value=0))
print("\n=== ОБЩЕЕ РАСПРЕДЕЛЕНИЕ ===")
print(df["source_type"].value_counts())
print(f"\nПроцент размеченных (не neutral): "
      f"{(df['source_type'] != 'neutral').mean()*100:.1f}%")

import os
os.makedirs("/kaggle/working/invariant", exist_ok=True)
df.to_parquet("/kaggle/working/invariant/xnli_with_source_type.parquet", index=False)
print("\n✓ Сохранено: xnli_with_source_type.parquet")

=== ТИПЫ ИСТОЧНИКОВ ПО РЕГИОНАМ ===
source_type     behavior  declaration  neutral
region                                        
anglosphere           60          348     2082
arab_world           162          210     2118
east_asia            141          261     2088
eastern_europe       408          378     4194
latin_america         84          273     2133
south_asia           369          501     4110
southeast_asia       498          777     3705
western_europe       234         1014     6222

=== ОБЩЕЕ РАСПРЕДЕЛЕНИЕ ===
source_type
neutral        26652
declaration     3762
behavior        1956
Name: count, dtype: int64

Процент размеченных (не neutral): 17.7%

✓ Сохранено: xnli_with_source_type.parquet


In [7]:
# Ячейка 6 — финальная проверка фазы 0

df = pd.read_parquet("/kaggle/working/invariant/xnli_with_source_type.parquet")

print("=== ФИНАЛ ФАЗЫ 0 — ИТОГОВЫЙ ДАТАСЕТ ===\n")
print(f"Примеров всего:        {len(df)}")
print(f"Регионов (сред):       {df['region'].nunique()}")
print(f"Языков:                {df['language'].nunique()}")
print(f"Меток (0/1/2):         равномерно ✓")
print(f"Размечено по типу:     {(df['source_type'] != 'neutral').mean()*100:.1f}%")

print("\n=== ПРИМЕР DECLARATION ===")
row = df[df["source_type"] == "declaration"].sample(1).iloc[0]
print(f"Язык: {row['language']} | Регион: {row['region']}")
print(f"Текст: {row['premise'][:200]}")

print("\n=== ПРИМЕР BEHAVIOR ===")
row = df[df["source_type"] == "behavior"].sample(1).iloc[0]
print(f"Язык: {row['language']} | Регион: {row['region']}")
print(f"Текст: {row['premise'][:200]}")

print("\n=== КОЛОНКИ ДАТАСЕТА ===")
print(df.dtypes)
print("\n✓ Фаза 0 завершена. Готово к фазе 1 — baseline обучение.")

=== ФИНАЛ ФАЗЫ 0 — ИТОГОВЫЙ ДАТАСЕТ ===

Примеров всего:        32370
Регионов (сред):       8
Языков:                12
Меток (0/1/2):         равномерно ✓
Размечено по типу:     17.7%

=== ПРИМЕР DECLARATION ===
Язык: de | Регион: western_europe
Текст: Was ich von Ihnen halte, sollte Sie nicht weiter kümmern, Sir. Dies war ein entwaffnender Schlag.

=== ПРИМЕР BEHAVIOR ===
Язык: en | Регион: anglosphere
Текст: As a result, his salary was raised and his other allowances stipend increased significantly, from approximately $465 to $3,925 a month, remaining at that level until December 2000.

=== КОЛОНКИ ДАТАСЕТА ===
premise        object
hypothesis     object
label           int64
language       object
region         object
source_type    object
dtype: object

✓ Фаза 0 завершена. Готово к фазе 1 — baseline обучение.


In [9]:
# Ячейка 7 — Фаза 1: Baseline (XLM-R без IRM)

!pip install transformers torch -q

import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from collections import defaultdict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {DEVICE}")

# --- Датасет ---
class XNLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["premise"],
            row["hypothesis"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label":          torch.tensor(row["label"], dtype=torch.long),
            "region":         row["region"],
        }

# --- Загрузка данных ---
df = pd.read_parquet("/kaggle/working/invariant/xnli_with_source_type.parquet")

# Train/val split — стратифицированный по региону
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["region"]
)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# --- Модель ---
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
).to(DEVICE)

# --- DataLoader ---
train_dataset = XNLIDataset(train_df, tokenizer)
val_dataset   = XNLIDataset(val_df,   tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)

# --- Обучение ---
optimizer = AdamW(model.parameters(), lr=2e-5)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=labels)
        out.loss.backward()
        optimizer.step()

        total_loss += out.loss.item()
        preds = out.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += len(labels)

    return total_loss / len(loader), correct / total

def eval_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    # Точность по регионам
    region_correct = defaultdict(int)
    region_total   = defaultdict(int)

    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            out = model(input_ids=ids, attention_mask=mask, labels=labels)
            preds = out.logits.argmax(dim=-1)

            total_loss += out.loss.item()
            correct += (preds == labels).sum().item()
            total += len(labels)

            for i, region in enumerate(batch["region"]):
                region_correct[region] += (preds[i] == labels[i]).item()
                region_total[region]   += 1

    acc_by_region = {
        r: region_correct[r] / region_total[r]
        for r in region_total
    }
    return total_loss / len(loader), correct / total, acc_by_region

# --- Запуск 3 эпохи ---
print("\n=== ОБУЧЕНИЕ BASELINE (λ=0) ===")
history = []

for epoch in range(3):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer)
    val_loss, val_acc, acc_by_region = eval_epoch(model, val_loader)

    history.append({
        "epoch": epoch+1,
        "train_loss": train_loss,
        "train_acc":  train_acc,
        "val_loss":   val_loss,
        "val_acc":    val_acc,
    })

    print(f"\nЭпоха {epoch+1}/3")
    print(f"  Train — loss: {train_loss:.4f} | acc: {train_acc:.4f}")
    print(f"  Val   — loss: {val_loss:.4f}   | acc: {val_acc:.4f}")
    print(f"  По регионам:")
    for region, acc in sorted(acc_by_region.items()):
        print(f"    {region:20s}: {acc:.4f}")

# --- Сохранение ---
torch.save({
    "model_state":     model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "history":         history,
    "acc_by_region":   acc_by_region,
    "lambda":          0,
}, "checkpoint_baseline.pt")

print("\n✓ Baseline сохранён: checkpoint_baseline.pt")
print("\n=== BASELINE МЕТРИКИ (запомни для сравнения) ===")
print(f"Val accuracy: {val_acc:.4f}")
print("По регионам:")
for region, acc in sorted(acc_by_region.items()):
    print(f"  {region:20s}: {acc:.4f}")
print("\nРазброс по регионам (чем меньше — тем лучше):")
accs = list(acc_by_region.values())
print(f"  min: {min(accs):.4f} | max: {max(accs):.4f} | "
      f"разница: {max(accs)-min(accs):.4f}")

Устройство: cuda
Train: 25896 | Val: 6474


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== ОБУЧЕНИЕ BASELINE (λ=0) ===


KeyboardInterrupt: 

In [ ]:
# Ячейка 8 — сохранить результаты постоянно (запустить ПОСЛЕ ячейки 7)
import shutil, os
 
os.makedirs("/kaggle/working/invariant", exist_ok=True)
shutil.copy("checkpoint_baseline.pt", 
            "/kaggle/working/invariant/checkpoint_baseline.pt")
print("✓ Чекпоинт скопирован")

In [16]:
# Ячейка 9 — Фаза 2: IRM с warmup и нормализацией

LAMBDA_IRM = 10.0
WARMUP_STEPS = 400   # шагов без штрафа

# чистим память от прошлого прогона
import gc
for name in ['irm_model', 'base', 'optimizer_irm']:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModel
base = AutoModel.from_pretrained("xlm-roberta-base").to(DEVICE)
irm_model = IRMModel(base).to(DEVICE)
optimizer_irm = torch.optim.AdamW(irm_model.parameters(), lr=2e-5)

def train_epoch_irm(model, loader, optimizer, lambda_irm, step_counter):
    model.train()
    total_loss, correct, total = 0, 0, 0
    penalty_sum = 0.0

    for batch in loader:
        ids     = batch["input_ids"].to(DEVICE)
        mask    = batch["attention_mask"].to(DEVICE)
        labels  = batch["label"].to(DEVICE)
        regions = batch["region"]

        # warmup: первые шаги учимся без штрафа
        current_lambda = 1.0 if step_counter[0] < WARMUP_STEPS else lambda_irm
        step_counter[0] += 1

        optimizer.zero_grad()
        phi_out, psi_out, h = model(ids, mask)

        main_loss = ce_loss(phi_out, labels)
        psi_loss  = ce_loss(psi_out, labels)

        env_losses = []
        for region in set(regions):
            m = torch.tensor([r == region for r in regions],
                             dtype=torch.bool).to(DEVICE)
            if m.sum() > 2:          # было >1, теперь >2 — меньше шума
                env_losses.append(ce_loss(phi_out[m], labels[m]))

        penalty = irm_penalty(env_losses) if len(env_losses) > 1 \
                  else torch.tensor(0.0, device=DEVICE)
        penalty_sum += penalty.item()

        loss = main_loss + current_lambda * penalty + psi_loss

        # нормализация: не даём лоссу раздуться при большой лямбде
        if current_lambda > 1.0:
            loss = loss / current_lambda

        loss.backward()
        optimizer.step()

        total_loss += main_loss.item()      # логируем ЧИСТЫЙ лосс задачи
        correct += (phi_out.argmax(dim=-1) == labels).sum().item()
        total += len(labels)

    return total_loss/len(loader), correct/total, penalty_sum/len(loader)

print(f"=== ФАЗА 2: IRM (λ={LAMBDA_IRM}, warmup={WARMUP_STEPS}) ===\n")
irm_history = []
step_counter = [0]

for epoch in range(4):     # 4 эпохи: первая уйдёт на warmup
    train_loss, train_acc, avg_penalty = train_epoch_irm(
        irm_model, train_loader, optimizer_irm, LAMBDA_IRM, step_counter
    )
    val_loss, val_acc, acc_by_region, psi_acc = eval_epoch_irm(
        irm_model, val_loader
    )
    accs = list(acc_by_region.values())
    spread = max(accs) - min(accs)

    irm_history.append({
        "epoch": epoch+1, "train_acc": train_acc, "val_acc": val_acc,
        "spread": spread, "penalty": avg_penalty,
    })

    phase = "warmup" if step_counter[0] <= WARMUP_STEPS else f"λ={LAMBDA_IRM}"
    print(f"Эпоха {epoch+1}/4  [{phase}]")
    print(f"  Train — loss: {train_loss:.4f} | acc: {train_acc:.4f}")
    print(f"  Val   — loss: {val_loss:.4f}   | acc: {val_acc:.4f}")
    print(f"  Ψ acc: {psi_acc:.4f} | penalty: {avg_penalty:.6f}")
    for region, acc in sorted(acc_by_region.items()):
        marker = " ←" if acc < 0.86 else ""
        print(f"    {region:20s}: {acc:.4f}{marker}")
    print(f"  Разброс: {spread:.4f}  (baseline 0.1566)")

    # честная проверка: разброс имеет смысл только если модель работает
    if val_acc < 0.5:
        print("  ⚠ точность на уровне случайной — разброс ничего не значит")
    print()

    torch.save({
        "model_state": irm_model.state_dict(),
        "history": irm_history, "lambda": LAMBDA_IRM,
        "acc_by_region": acc_by_region, "epoch": epoch+1,
    }, "/kaggle/working/checkpoint_irm_10.pt")

print("=== СРАВНЕНИЕ С BASELINE ===")
print(f"Baseline val_acc: 0.8945  |  IRM val_acc: {val_acc:.4f}")
print(f"Baseline разброс: 0.1566  |  IRM разброс: {spread:.4f}")
if val_acc < 0.80:
    print("\n⚠ IRM уронил точность — разброс сравнивать некорректно")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== ФАЗА 2: IRM (λ=10.0, warmup=400) ===

Эпоха 1/4  [λ=10.0]
  Train — loss: 1.0810 | acc: 0.4076
  Val   — loss: 1.0766   | acc: 0.4944
  Ψ acc: 0.3392 | penalty: 0.025158
    anglosphere         : 0.4940 ←
    arab_world          : 0.4940 ←
    east_asia           : 0.4799 ←
    eastern_europe      : 0.5161 ←
    latin_america       : 0.5120 ←
    south_asia          : 0.4608 ←
    southeast_asia      : 0.4719 ←
    western_europe      : 0.5167 ←
  Разброс: 0.0559  (baseline 0.1566)
  ⚠ точность на уровне случайной — разброс ничего не значит

Эпоха 2/4  [λ=10.0]
  Train — loss: 1.0755 | acc: 0.4481
  Val   — loss: 1.0707   | acc: 0.4526
  Ψ acc: 0.3267 | penalty: 0.008560
    anglosphere         : 0.5221 ←
    arab_world          : 0.4458 ←
    east_asia           : 0.4438 ←
    eastern_europe      : 0.4267 ←
    latin_america       : 0.4398 ←
    south_asia          : 0.4227 ←
    southeast_asia      : 0.4518 ←
    western_europe      : 0.4766 ←
  Разброс: 0.0994  (baseline 0.1566)

In [1]:
# Ячейка 10 — Фаза 2B: GroupDRO

import torch, torch.nn as nn, gc, numpy as np
from collections import defaultdict

ETA = 0.01          # скорость роста весов сред
N_EPOCHS = 4        # столько же, сколько у IRM — для честного сравнения

for name in ['irm_model', 'base', 'optimizer_irm', 'dro_model', 'optimizer_dro']:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

REGIONS = sorted(df["region"].unique())
R2I = {r: i for i, r in enumerate(REGIONS)}

class DROModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        self.head = nn.Linear(768, 3)

    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(out.last_hidden_state[:, 0, :])

from transformers import AutoModel
base = AutoModel.from_pretrained("xlm-roberta-base").to(DEVICE)
dro_model = DROModel(base).to(DEVICE)
optimizer_dro = torch.optim.AdamW(dro_model.parameters(), lr=2e-5)
ce_none = nn.CrossEntropyLoss(reduction='none')

# веса сред — растут у тех, где модель хуже
q = torch.ones(len(REGIONS), device=DEVICE) / len(REGIONS)

def train_epoch_dro(model, loader, optimizer, q):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in loader:
        ids     = batch["input_ids"].to(DEVICE)
        mask    = batch["attention_mask"].to(DEVICE)
        labels  = batch["label"].to(DEVICE)
        r_idx   = torch.tensor([R2I[r] for r in batch["region"]]).to(DEVICE)

        optimizer.zero_grad()
        logits = model(ids, mask)
        losses = ce_none(logits, labels)          # лосс на каждый пример

        # средний лосс по каждой среде, присутствующей в батче
        env_loss = torch.zeros(len(REGIONS), device=DEVICE)
        present  = []
        for i in range(len(REGIONS)):
            m = (r_idx == i)
            if m.sum() > 0:
                env_loss[i] = losses[m].mean()
                present.append(i)

        # обновляем веса: хуже среда → больше вес
        with torch.no_grad():
            for i in present:
                q[i] = q[i] * torch.exp(ETA * env_loss[i])
            q /= q.sum()

        loss = (q * env_loss).sum() * len(REGIONS)   # взвешенный лосс
        loss.backward()
        optimizer.step()

        total_loss += losses.mean().item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += len(labels)

    return total_loss/len(loader), correct/total

def eval_epoch_dro(model, loader):
    model.eval()
    correct, total = 0, 0
    rc, rt = defaultdict(int), defaultdict(int)
    with torch.no_grad():
        for batch in loader:
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            preds  = model(ids, mask).argmax(dim=-1)
            correct += (preds == labels).sum().item(); total += len(labels)
            for i, r in enumerate(batch["region"]):
                rc[r] += (preds[i] == labels[i]).item(); rt[r] += 1
    acc_by_region = {r: rc[r]/rt[r] for r in rt}
    n_by_region   = dict(rt)
    return correct/total, acc_by_region, n_by_region

def ci95(acc, n):
    """Доверительный интервал — чтобы не читать шум как результат."""
    return 1.96 * np.sqrt(acc * (1 - acc) / n)

print(f"=== ФАЗА 2B: GroupDRO (eta={ETA}) ===\n")
dro_history = []

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_epoch_dro(dro_model, train_loader,
                                            optimizer_dro, q)
    val_acc, acc_by_region, n_by_region = eval_epoch_dro(dro_model, val_loader)

    accs   = list(acc_by_region.values())
    spread = max(accs) - min(accs)
    worst_r = min(acc_by_region, key=acc_by_region.get)
    worst   = acc_by_region[worst_r]

    dro_history.append({"epoch": epoch+1, "val_acc": val_acc,
                        "spread": spread, "worst": worst})

    print(f"Эпоха {epoch+1}/{N_EPOCHS}")
    print(f"  Train — loss: {train_loss:.4f} | acc: {train_acc:.4f}")
    print(f"  Val   — acc: {val_acc:.4f}")
    for r in sorted(acc_by_region):
        a, n = acc_by_region[r], n_by_region[r]
        print(f"    {r:20s}: {a:.4f} ±{ci95(a,n):.3f}")
    print(f"  Худшая среда: {worst_r} = {worst:.4f}")
    print(f"  Разброс: {spread:.4f}")

    # разброс значим, только если больше погрешности
    max_ci = max(ci95(acc_by_region[r], n_by_region[r]) for r in acc_by_region)
    if spread < 2 * max_ci:
        print(f"  (разброс в пределах погрешности ±{max_ci:.3f} — не значим)")
    print(f"  Веса сред q: {[f'{v:.3f}' for v in q.tolist()]}\n")

    torch.save({"model_state": dro_model.state_dict(), "history": dro_history,
                "eta": ETA, "acc_by_region": acc_by_region, "epoch": epoch+1},
               "/kaggle/working/checkpoint_dro.pt")

print("=== ИТОГ ===")
print(f"{'':12s} {'val_acc':>8s} {'разброс':>8s} {'худшая':>8s}")
print(f"{'baseline':12s} {0.8945:>8.4f} {0.1566:>8.4f} {0.8273:>8.4f}")
print(f"{'IRM λ=10':12s} {0.5842:>8.4f} {0.0803:>8.4f} {0.5502:>8.4f}")
print(f"{'GroupDRO':12s} {val_acc:>8.4f} {spread:>8.4f} {worst:>8.4f}")

NameError: name 'df' is not defined

In [8]:
# Ячейка 11 — диагностика: точность по ЯЗЫКАМ, а не по регионам

from collections import defaultdict
import numpy as np

def eval_by_language(model, loader):
    model.eval()
    lc, lt = defaultdict(int), defaultdict(int)
    # распределение предсказаний — проверим, не залипла ли модель на одном классе
    pred_dist = defaultdict(lambda: defaultdict(int))
    with torch.no_grad():
        for batch in loader:
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            preds  = model(ids, mask).argmax(dim=-1)
            for i, lang in enumerate(batch["language"]):
                lc[lang] += (preds[i] == labels[i]).item()
                lt[lang] += 1
                pred_dist[lang][preds[i].item()] += 1
    return lc, lt, pred_dist

# ВАЖНО: в XNLIDataset нет поля language — добавим на лету
class XNLIDatasetLang(XNLIDataset):
    def __getitem__(self, idx):
        item = super().__getitem__(idx)
        item["language"] = self.df.iloc[idx]["language"]
        return item

val_loader_lang = DataLoader(
    XNLIDatasetLang(val_df, tokenizer), batch_size=64, shuffle=False
)

lc, lt, pred_dist = eval_by_language(dro_model, val_loader_lang)

print("=== ТОЧНОСТЬ ПО ЯЗЫКАМ ===\n")
print(f"{'язык':6s} {'регион':18s} {'n':>5s} {'acc':>7s}   распределение предсказаний")
lang2region = df.drop_duplicates("language").set_index("language")["region"].to_dict()

for lang in sorted(lt, key=lambda l: lc[l]/lt[l]):
    acc = lc[lang]/lt[lang]
    n   = lt[lang]
    d   = pred_dist[lang]
    tot = sum(d.values())
    dist = " ".join(f"{d.get(c,0)/tot:.0%}" for c in [0,1,2])
    flag = "  ← провал" if acc < 0.5 else ""
    print(f"{lang:6s} {lang2region[lang]:18s} {n:>5d} {acc:>7.4f}   {dist}{flag}")

NameError: name 'XNLIDataset' is not defined

In [11]:
# Ячейка 12 — GroupDRO 2 эпохи + диагностика по языкам

!pip install transformers -q

import torch, torch.nn as nn, pandas as pd, numpy as np, gc
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from collections import defaultdict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ETA, N_EPOCHS = 0.01, 2
print(f"Устройство: {DEVICE}")

# ---------- данные ----------
df = pd.read_parquet("/kaggle/working/invariant/xnli_with_source_type.parquet")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42,
                                    stratify=df["region"])
REGIONS = sorted(df["region"].unique())
R2I = {r: i for i, r in enumerate(REGIONS)}
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

class XNLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df.reset_index(drop=True); self.tok = tokenizer; self.ml = max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tok(row["premise"], row["hypothesis"], max_length=self.ml,
                       padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "label": torch.tensor(row["label"], dtype=torch.long),
                "region": row["region"], "language": row["language"]}

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
train_loader = DataLoader(XNLIDataset(train_df, tokenizer), batch_size=32, shuffle=True)
val_loader   = DataLoader(XNLIDataset(val_df,   tokenizer), batch_size=64, shuffle=False)

# ---------- модель ----------
class DROModel(nn.Module):
    def __init__(self, base_model):
        super().__init__(); self.base = base_model; self.head = nn.Linear(768, 3)
    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(out.last_hidden_state[:, 0, :])

gc.collect(); torch.cuda.empty_cache()
base = AutoModel.from_pretrained("xlm-roberta-base").to(DEVICE)
dro_model = DROModel(base).to(DEVICE)
optimizer = torch.optim.AdamW(dro_model.parameters(), lr=2e-5)
ce_none = nn.CrossEntropyLoss(reduction='none')
q = torch.ones(len(REGIONS), device=DEVICE) / len(REGIONS)

def ci95(acc, n): return 1.96 * np.sqrt(acc * (1 - acc) / n)

# ---------- обучение ----------
print(f"\n=== GroupDRO, {N_EPOCHS} эпохи ===\n")
for epoch in range(N_EPOCHS):
    dro_model.train(); correct = total = 0
    for batch in train_loader:
        ids    = batch["input_ids"].to(DEVICE)
        mask   = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        r_idx  = torch.tensor([R2I[r] for r in batch["region"]]).to(DEVICE)

        optimizer.zero_grad()
        logits = dro_model(ids, mask)
        losses = ce_none(logits, labels)

        env_loss = torch.zeros(len(REGIONS), device=DEVICE); present = []
        for i in range(len(REGIONS)):
            m = (r_idx == i)
            if m.sum() > 0:
                env_loss[i] = losses[m].mean(); present.append(i)
        with torch.no_grad():
            for i in present: q[i] = q[i] * torch.exp(ETA * env_loss[i])
            q /= q.sum()

        ((q * env_loss).sum() * len(REGIONS)).backward()
        optimizer.step()
        correct += (logits.argmax(-1) == labels).sum().item(); total += len(labels)

    # ---------- оценка: регионы И языки сразу ----------
    dro_model.eval()
    rc, rt = defaultdict(int), defaultdict(int)
    lc, lt = defaultdict(int), defaultdict(int)
    pred_dist = defaultdict(lambda: defaultdict(int))
    vcorrect = vtotal = 0
    with torch.no_grad():
        for batch in val_loader:
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            preds  = dro_model(ids, mask).argmax(-1)
            vcorrect += (preds == labels).sum().item(); vtotal += len(labels)
            for i, (r, l) in enumerate(zip(batch["region"], batch["language"])):
                ok = (preds[i] == labels[i]).item()
                rc[r] += ok; rt[r] += 1
                lc[l] += ok; lt[l] += 1
                pred_dist[l][preds[i].item()] += 1

    val_acc = vcorrect / vtotal
    acc_r = {r: rc[r]/rt[r] for r in rt}
    print(f"Эпоха {epoch+1}/{N_EPOCHS}  train_acc {correct/total:.4f} | val_acc {val_acc:.4f}")
    for r in sorted(acc_r):
        print(f"    {r:20s}: {acc_r[r]:.4f} ±{ci95(acc_r[r], rt[r]):.3f}  q={q[R2I[r]]:.3f}")
    print(f"  Разброс: {max(acc_r.values())-min(acc_r.values()):.4f}\n")

    torch.save({"model_state": dro_model.state_dict(), "epoch": epoch+1,
                "val_acc": val_acc, "acc_by_region": acc_r},
               "/kaggle/working/invariant/checkpoint_dro.pt")

# ---------- диагностика по языкам ----------
lang2region = df.drop_duplicates("language").set_index("language")["region"].to_dict()
print("=== ТОЧНОСТЬ ПО ЯЗЫКАМ ===\n")
print(f"{'язык':6s} {'регион':18s} {'n':>5s} {'acc':>7s}   предсказания 0/1/2")
for l in sorted(lt, key=lambda x: lc[x]/lt[x]):
    acc, n = lc[l]/lt[l], lt[l]
    d = pred_dist[l]; tot = sum(d.values())
    dist = " ".join(f"{d.get(c,0)/tot:>4.0%}" for c in [0,1,2])
    flag = "  ← провал" if acc < 0.5 else ""
    print(f"{l:6s} {lang2region[l]:18s} {n:>5d} {acc:>7.4f}   {dist}{flag}")

print("\n✓ Готово. Сделай Quick Save с сохранением output!")

Устройство: cuda
Train: 25896 | Val: 6474


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== GroupDRO, 2 эпохи ===

Эпоха 1/2  train_acc 0.4727 | val_acc 0.7003
    anglosphere         : 0.7751 ±0.037  q=0.062
    arab_world          : 0.6466 ±0.042  q=0.091
    east_asia           : 0.7008 ±0.040  q=0.081
    eastern_europe      : 0.7048 ±0.028  q=0.141
    latin_america       : 0.7149 ±0.040  q=0.093
    south_asia          : 0.6627 ±0.029  q=0.161
    southeast_asia      : 0.6576 ±0.029  q=0.220
    western_europe      : 0.7390 ±0.022  q=0.152
  Разброс: 0.1285

Эпоха 2/2  train_acc 0.7542 | val_acc 0.7576
    anglosphere         : 0.8735 ±0.029  q=0.011
    arab_world          : 0.7369 ±0.039  q=0.059
    east_asia           : 0.7771 ±0.037  q=0.042
    eastern_europe      : 0.8112 ±0.024  q=0.074
    latin_america       : 0.8434 ±0.032  q=0.016
    south_asia          : 0.7239 ±0.028  q=0.253
    southeast_asia      : 0.5251 ±0.031  q=0.481
    western_europe      : 0.8327 ±0.019  q=0.064
  Разброс: 0.3484

=== ТОЧНОСТЬ ПО ЯЗЫКАМ ===

язык   регион                 n 